In [17]:
import numpy as np
import pandas as pd
import ast, pickle, os
from nltk.stem import PorterStemmer
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

ps = PorterStemmer()
model = SentenceTransformer('all-MiniLM-L6-v2')  # downloads ~80MB once
print("Model loaded ✅")

c:\Users\priya\anaconda3\envs\popcorn\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4512.45it/s]


Model loaded ✅


In [18]:
DATA_DIR = r"D:\CODE\Projects\PopcornPicks"
movies  = pd.read_csv(os.path.join(DATA_DIR, 'tmdb_5000_movies.csv'))
credits = pd.read_csv(os.path.join(DATA_DIR, 'tmdb_5000_credits.csv'))
movies  = movies.merge(credits, on='title')
movies  = movies[['movie_id','title','overview','genres','keywords',
                   'cast','crew','vote_average','vote_count','release_date']]
movies.dropna(inplace=True)

In [19]:
def convert(text):      return [i['name'] for i in ast.literal_eval(text)]
def convert3(text):     return [i['name'] for i in ast.literal_eval(text)][:3]
def fetch_director(t):  return [i['name'] for i in ast.literal_eval(t) if i['job']=='Director']
def collapse(lst):      return [i.replace(" ","") for i in lst]
def stem(text):         return " ".join([ps.stem(w) for w in text.split()])

movies['genres']   = movies['genres'].apply(convert).apply(collapse)
movies['keywords'] = movies['keywords'].apply(convert).apply(collapse)
movies['cast']     = movies['cast'].apply(convert3).apply(collapse)
movies['crew']     = movies['crew'].apply(fetch_director).apply(collapse)
movies['overview'] = movies['overview'].apply(lambda x: x.split())

movies['tags'] = (movies['overview'] + movies['genres'] +
                  movies['keywords'] + movies['cast'] + movies['crew'])

new_df = movies[['movie_id','title','tags','genres','cast',
                  'crew','vote_average','vote_count','release_date']].copy()

new_df['tags']   = new_df['tags'].apply(lambda x: " ".join(x).lower()).apply(stem)
new_df['genres'] = new_df['genres'].apply(lambda x: " ".join(x))
new_df['cast']   = new_df['cast'].apply(lambda x: " ".join(x))
new_df['crew']   = new_df['crew'].apply(lambda x: " ".join(x))
new_df['year']   = pd.to_datetime(new_df['release_date'], errors='coerce').dt.year

In [20]:
print("Generating embeddings... (takes ~2-3 mins first time)")
tags_list   = new_df['tags'].tolist()
embeddings  = model.encode(tags_list, show_progress_bar=True, batch_size=64)

# Content similarity from sentence transformers
content_sim = cosine_similarity(embeddings)

# Hybrid: 85% semantic + 15% popularity
new_df['popularity_score'] = (
    new_df['vote_average'] * np.log1p(new_df['vote_count'])
)
new_df['popularity_score'] /= new_df['popularity_score'].max()

pop_matrix = np.outer(new_df['popularity_score'].values,
                      new_df['popularity_score'].values)
similarity = 0.85 * content_sim + 0.15 * pop_matrix

print(f"✅ Embeddings shape: {embeddings.shape}")
print(f"✅ Similarity matrix: {similarity.shape}")

Generating embeddings... (takes ~2-3 mins first time)


Batches: 100%|██████████| 76/76 [01:09<00:00,  1.09it/s]


✅ Embeddings shape: (4805, 384)
✅ Similarity matrix: (4805, 4805)


In [21]:
pickle.dump(new_df,     open('movie_list.pkl', 'wb'))
pickle.dump(similarity, open('similarity.pkl', 'wb'))
pickle.dump(embeddings, open('embeddings.pkl', 'wb'))  # save for future use
print("✅ All files saved!")

✅ All files saved!
